# 04 MLP 모델 학습

- MLP 기반의 이진 분류기를 학습한다.
- 데이터 누수를 방지하기 위해 PCA, 메타데이터 스케일링, SMOTE, MLP 학습을 5-Fold 교차검증의 각 fold 내부에서 수행한다.
- 전체 train 데이터에 PCA와 스케일러를 먼저 학습한 뒤 fold를 나누지 않는다.
- `rating`은 이후 별점 정제 대상이므로 이벤트 판별 입력 feature에서 제외한다.
- 각 MLP 설정마다 5-fold 교차검증을 수행하고, 각 fold 안에서 scaler, PCA, SMOTE를 적용한 뒤 모델을 학습한다. 이후 fold별 검증 성능을 평균내고, 평균 F1-score가 가장 높은 설정을 최적 설정으로 선택한다.
- 이미 `outputs/proposed_mlp_final_model.joblib` 등 필수 산출물이 있고 feature schema가 맞으면 기존 04번 학습 결과를 재사용한다.
- 다시 학습하고 싶으면 `FORCE_RETRAIN_MLP = True`로 바꾼다.
- **즉 현재 코드는 여러 MLP에 대해서 테스트 후 최적의 설정을 찾아서 해당 모델을 학습시킨다.**

### Fold 내부 전처리 및 SMOTE 적용 방식

1. raw KcBERT 임베딩과 raw 메타데이터를 train / validation / test로 분리한다.
2. train 데이터 안에서만 5-Fold 교차검증을 수행한다.
3. 각 fold의 학습 데이터에만 PCA와 StandardScaler를 fit한다.
4. 변환된 학습 fold에만 SMOTE를 적용한다.
5. fold 검증 데이터, validation 데이터, test 데이터는 원본 클래스 분포를 유지한다.
6. fold별 성능을 평균내어 후보 MLP 설정을 비교한다.
- 전처리와 SMOTE를 fold 내부에서만 fit해야 validation/test 정보가 train에 섞이는 누수를 막을 수 있다.

## 1. 학습용 라이브러리 로드

In [1]:
import json
import os
from itertools import product

import joblib
import numpy as np
import pandas as pd
from IPython.display import display
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline
from sklearn.compose import ColumnTransformer
from sklearn.decomposition import PCA
from sklearn.metrics import (
    average_precision_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler

## 2. 데이터 로드 및 입력/라벨 분리

- 02번에서 생성한 `reviews_embeddings_extract.csv`를 사용한다.
- PCA와 메타데이터 스케일링을 CV fold 내부에서 학습하기 위해 raw feature 상태로 로드한다.
- train / validation / test를 70% / 15% / 15%로 분리한다.
- `rating`은 이벤트 판별 feature에서 제외한다.
- raw feature 상태로 split해야 PCA, scaler, SMOTE를 각 fold 안에서만 학습할 수 있기 때문이다.
- 04번 학습 산출물이 이미 있고 feature schema가 현재 입력과 같으면 CV와 최종 학습을 재사용한다.

In [2]:
SEED = 42
TARGET_COL = 'label'
TEXT_COL = 'cleaned_review_text'

raw_df = pd.read_csv('csv/reviews_embeddings_extract.csv')

emb_cols = [f'kcbert_{i}' for i in range(768)]
meta_cols = ['text_length', 'emoji_count', 'photo_count']
feature_cols = emb_cols + meta_cols

X_all = raw_df[feature_cols].copy()
y_all = raw_df[TARGET_COL].astype(int)

X_train_all, X_temp, y_train_all, y_temp = train_test_split(
    X_all,
    y_all,
    test_size=0.3,
    random_state=SEED,
    stratify=y_all,
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.5,
    random_state=SEED,
    stratify=y_temp,
)

print('전체 데이터:', raw_df.shape)
print('학습용 데이터:', X_train_all.shape)
print('검증용 데이터:', X_val.shape)
print('테스트용 데이터:', X_test.shape)
print('입력 feature 수:', len(feature_cols))
print('사용 메타데이터 컬럼:', meta_cols)
print('학습 데이터 라벨 분포, 0=일반 리뷰, 1=이벤트 리뷰')
print(y_train_all.value_counts().sort_index().to_dict())

전체 데이터: (8841, 782)
학습용 데이터: (6188, 771)
검증용 데이터: (1326, 771)
테스트용 데이터: (1327, 771)
입력 feature 수: 771
사용 메타데이터 컬럼: ['text_length', 'emoji_count', 'photo_count']
학습 데이터 라벨 분포, 0=일반 리뷰, 1=이벤트 리뷰
{0: 3983, 1: 2205}


/var/folders/wq/j2lqqp7n33dgrwj1r4q0nl9m0000gn/T/ipykernel_74697/2170499664.py:5: DtypeWarning: Columns (0: store_name) have mixed types. Specify dtype option on import or set low_memory=False.
  raw_df = pd.read_csv('csv/reviews_embeddings_extract.csv')


## 3. 후보 MLP 모델 정의

- 최종 분류 모델은 Scikit-learn `MLPClassifier`를 사용한다.
- 입력 데이터는 raw KcBERT CLS 임베딩 768차원과 `text_length`, `emoji_count`, `photo_count` 메타데이터이다.
- `ColumnTransformer`를 사용해 KcBERT 임베딩에는 PCA를, 메타데이터에는 StandardScaler를 적용한다.
    - KcBERT 임베딩에는 PCA를 적용하고, 메타데이터에는 StandardScaler를 적용한 뒤, 두 결과를 하나의 feature로 결합하기 위해 ColumnTransformer를 사용합니다.
- PCA와 StandardScaler는 전체 데이터에 미리 fit하지 않고, 각 CV fold 내부의 학습 데이터에서만 fit한다.
- SMOTE는 전처리 이후 학습 fold에만 적용하고, 검증 fold에는 적용하지 않는다.
    - PCA와 StandardScaler는 fit 과정에서 데이터의 평균, 분산, 주성분 구조를 학습하므로 전체 데이터에 미리 학습하면 validation/test 분포 정보가 학습 과정에 섞일 수 있다.
    - SMOTE도 전체 데이터에 먼저 적용하면 검증 데이터와 유사한 합성 샘플이 학습 fold에 들어갈 수 있으므로, 전처리와 SMOTE는 각 fold의 학습 데이터에만 적용한다.
- 하나의 MLP 구조만 고정하지 않고, 은닉층 구조, optimizer/solver, learning rate, L2 정규화(alpha), activation, batch size를 함께 비교한다.
    - 이후 5-fold CV 평균 F1-score를 기준으로 가장 좋은 MLP 설정을 최종 후보 모델로 선택한다.

### **각 변경되는 모델 특성과 역할** (계속해서 업데이트 가능)
- hidden layer: MLP의 은닉층 구조이다. `(32,)`, `(64,)`, `(128,)`은 은닉층 1개 구조이고, `(128, 64)`는 은닉층 2개 구조이다. 값이 커질수록 더 복잡한 패턴을 학습할 수 있지만 과적합 위험도 커질 수 있다.
- optimizer/solver: 손실을 줄이기 위해 가중치를 업데이트하는 방식이다. `adam`은 일반적으로 빠르고 안정적인 최적화 방식이며, `sgd`는 기본적인 경사하강 방식이다. 계획서에는 Adam 중심으로 작성했지만, 실제 구현에서는 비교 범위를 넓히기 위해 `sgd`도 함께 평가한다.
- learning rate: 한 번 업데이트할 때 가중치를 얼마나 크게 바꿀지 정하는 값이다. `1e-3`은 비교적 빠르게 학습할 수 있고, `3e-4`는 더 작게 움직여 안정적으로 학습할 수 있다.
- L2 alpha: MLP의 L2 정규화 강도이다. 값이 작으면 모델이 더 자유롭게 학습하지만 과적합 위험이 커질 수 있고, 값이 크면 가중치가 과도하게 커지는 것을 막아 과적합을 줄일 수 있다.
- activation: 은닉층에서 사용하는 비선형 변환 함수이다. `relu`는 학습이 빠르고 일반적으로 많이 쓰이며, `tanh`는 출력을 -1에서 1 사이로 압축해 다른 표현 방식을 제공한다. 어떤 함수가 현재 데이터에 더 맞는지 확인하기 위해 비교한다.
    - 'relu'는 일반적으로 많이 쓰인다. (0 혹은 양수)
    - 'tanh'은 음수도 표현하기 때문에, relu와 비교하기 좋다.
- batch size: 한 번에 묶어서 학습하는 데이터 개수이다. `32`는 업데이트가 더 자주 일어나 세밀하게 학습될 수 있고, `64`는 더 안정적인 방향으로 학습될 수 있다.
- PCA 보존율: KcBERT 768차원 임베딩을 줄일 때 보존할 분산 정보의 비율이다. 현재는 90%를 보존하도록 설정해 의미 정보는 대부분 유지하면서 차원을 줄인다. 이 값은 `pca_n_components_grid`에서 확장할 수 있다.
- SMOTE k_neighbors: 소수 클래스 샘플을 새로 만들 때 참고하는 이웃 샘플 수이다. 현재는 기본값 5를 사용하며, 학습 fold 안에서만 적용해 클래스 불균형을 보정한다.

### 탐색 범위

- hidden layer: `(32,)`, `(64,)`, `(128,)`, `(128, 64)`
- optimizer/solver: `adam`, `sgd`
- learning rate: `1e-3`, `3e-4`
- L2 alpha: `1e-4`, `1e-3`, `1e-2`
- activation: `relu`, `tanh`
- batch size: `32`, `64`
- PCA 보존율: `0.90`
- SMOTE k_neighbors: `5`

총 후보 수는 다음과 같다.

`4 x 2 x 2 x 3 x 2 x 2 x 1 x 1 = 192개`

각 후보를 5-fold 교차검증으로 평가하므로 총 **`192 x 5 = 960개 fold 결과`**를 비교한다.

### 설정 선택 기준

- 각 후보 설정마다 5-fold CV를 수행한다.
- 각 fold에서는 학습 fold에만 PCA, StandardScaler, SMOTE를 fit한다.
- 검증 fold에는 학습 fold에서 fit된 전처리기를 transform만 적용한다.
- fold별 F1-score, PR-AUC, precision, recall, ROC-AUC를 계산한다.
- 후보별 5개 fold 성능을 평균낸 뒤, 평균 F1-score가 가장 높은 설정을 우선 선택한다.
- 평균 F1-score가 비슷하거나 같은 경우 PR-AUC를 보조 기준으로 사용한다.

수행계획서의 은닉층 구조, Adam optimizer 학습률, L2 정규화 최적화 요구를 충족하면서, 실제 구현에서는 solver, activation, batch size까지 비교 범위를 넓혀 현재 데이터에 더 적합한 MLP 설정을 찾기 위해서다.

In [3]:
hidden_layer_grid = [(32,), (64,), (128,), (128, 64)]
solver_grid = ['adam', 'sgd']
learning_rate_grid = [1e-3, 3e-4]
alpha_grid = [1e-4, 1e-3, 1e-2]
activation_grid = ['relu', 'tanh']
batch_size_grid = [32, 64]
pca_n_components_grid = [0.90]
smote_k_neighbors_grid = [5]
N_SPLITS = 5

model_configs = []
for hidden_layer_sizes, solver, learning_rate_init, alpha, activation, batch_size, pca_n_components, smote_k_neighbors in product(
    hidden_layer_grid,
    solver_grid,
    learning_rate_grid,
    alpha_grid,
    activation_grid,
    batch_size_grid,
    pca_n_components_grid,
    smote_k_neighbors_grid,
):
    hidden_name = '_'.join(str(size) for size in hidden_layer_sizes)
    pca_name = str(pca_n_components).replace('.', 'p')
    model_configs.append({
        'name': f"mlp_h{hidden_name}_{activation}_{solver}_lr{learning_rate_init:g}_alpha{alpha:g}_bs{batch_size}_pca{pca_name}_smote{smote_k_neighbors}",
        'hidden_layer_sizes': hidden_layer_sizes,
        'solver': solver,
        'learning_rate_init': learning_rate_init,
        'alpha': alpha,
        'activation': activation,
        'batch_size': batch_size,
        'pca_n_components': pca_n_components,
        'smote_k_neighbors': smote_k_neighbors,
    })

CV_GRID_SIZE = len(model_configs)
print(f'CV 후보 설정 수: {CV_GRID_SIZE}, fold 수: {N_SPLITS}, 총 fold 평가 수: {CV_GRID_SIZE * N_SPLITS}')

PROPOSED_MLP_OUTPUTS = {
    'model': 'outputs/proposed_mlp_final_model.joblib',
    'metrics_csv': 'outputs/proposed_mlp_metrics.csv',
    'metrics_json': 'outputs/proposed_mlp_metrics.json',
    'selected_config': 'outputs/proposed_mlp_selected_config.json',
    'cv_results': 'outputs/proposed_mlp_cv_results.csv',
    'cv_summary': 'outputs/proposed_mlp_cv_summary.csv',
}
FORCE_RETRAIN_MLP = False


def check_mlp_cache():
    if FORCE_RETRAIN_MLP:
        return False, 'FORCE_RETRAIN_MLP=True이므로 04번 MLP CV와 최종 학습을 새로 실행합니다.'

    missing_paths = [path for path in PROPOSED_MLP_OUTPUTS.values() if not os.path.exists(path)]
    if missing_paths:
        return False, f'04번 필수 산출물이 없어 새로 학습합니다: {missing_paths}'

    try:
        bundle = joblib.load(PROPOSED_MLP_OUTPUTS['model'])
    except Exception as exc:
        return False, f'저장된 MLP bundle을 로드할 수 없어 새로 학습합니다: {exc}'

    cached_feature_cols = bundle.get('feature_cols')
    if cached_feature_cols != feature_cols:
        return (
            False,
            '저장된 MLP bundle의 feature_cols가 현재 feature_cols와 달라 새로 학습합니다. '
            f'저장 feature 수: {0 if cached_feature_cols is None else len(cached_feature_cols)}, 현재 feature 수: {len(feature_cols)}'
        )

    try:
        with open(PROPOSED_MLP_OUTPUTS['selected_config'], 'r', encoding='utf-8') as f:
            cached_selected = json.load(f)
    except Exception as exc:
        return False, f'저장된 MLP 설정 파일을 로드할 수 없어 새로 학습합니다: {exc}'
    if cached_selected.get('cv_grid_size') != CV_GRID_SIZE or cached_selected.get('n_splits') != N_SPLITS:
        return (
            False,
            '저장된 MLP 탐색 설정이 현재 grid와 달라 새로 학습합니다. '
            f"저장 grid/fold: {cached_selected.get('cv_grid_size')}/{cached_selected.get('n_splits')}, "
            f'현재 grid/fold: {CV_GRID_SIZE}/{N_SPLITS}'
        )

    return True, f"기존 04번 MLP 산출물을 재사용합니다: {PROPOSED_MLP_OUTPUTS['model']}"


can_reuse_mlp, mlp_cache_message = check_mlp_cache()
print(mlp_cache_message)

cached_model_bundle = None
cached_metrics = None
cached_selected_config = None
if can_reuse_mlp:
    cached_model_bundle = joblib.load(PROPOSED_MLP_OUTPUTS['model'])
    cached_metrics = pd.read_csv(PROPOSED_MLP_OUTPUTS['metrics_csv'])
    with open(PROPOSED_MLP_OUTPUTS['selected_config'], 'r', encoding='utf-8') as f:
        cached_selected_config = json.load(f)
    cv_results = pd.read_csv(PROPOSED_MLP_OUTPUTS['cv_results'])
    cv_summary = pd.read_csv(PROPOSED_MLP_OUTPUTS['cv_summary'])


def make_preprocessor(random_state: int, pca_n_components: float, use_emb=True, use_meta=True):
    transformers = []
    if use_emb:
        transformers.append(('kcbert_pca', PCA(n_components=pca_n_components, random_state=random_state), emb_cols))
    if use_meta:
        transformers.append(('metadata_scaler', StandardScaler(), meta_cols))

    return ColumnTransformer(transformers=transformers, remainder='drop')


def make_model(config: dict, random_state: int) -> ImbPipeline:
    return ImbPipeline([
        ('preprocess', make_preprocessor(
            random_state=random_state,
            pca_n_components=config.get('pca_n_components', 0.90),
            use_emb=True,
            use_meta=True,
        )),
        ('smote', SMOTE(
            random_state=random_state,
            k_neighbors=config.get('smote_k_neighbors', 5),
        )),
        ('mlp', MLPClassifier(
            hidden_layer_sizes=config['hidden_layer_sizes'],
            activation=config.get('activation', 'relu'),
            solver=config['solver'],
            alpha=config['alpha'],
            batch_size=config.get('batch_size', 64),
            learning_rate_init=config['learning_rate_init'],
            max_iter=200,
            early_stopping=True,
            validation_fraction=0.15,
            n_iter_no_change=10,
            random_state=random_state,
        )),
    ])

CV 후보 설정 수: 192, fold 수: 5, 총 fold 평가 수: 960
기존 04번 MLP 산출물을 재사용합니다: outputs/proposed_mlp_final_model.joblib


## 4. SMOTE + 5-Fold 교차검증 (시간 오래 걸릴 수 있음)

- 5-Fold 교차검증은 train 데이터 안에서 후보 MLP 모델들의 성능을 비교하기 위한 학습/검증 단계이다.
- 먼저 train 데이터를 5개의 fold로 나눈 뒤, 각 반복에서 4개 fold는 학습용, 1개 fold는 검증용으로 사용한다.
- 클래스 불균형 보정을 위해 각 반복의 학습 fold에만 SMOTE를 적용한다.
- 검증 fold에는 SMOTE를 적용하지 않고, 원본 분포 그대로 성능을 평가한다.
- 앞서 정의한 여러 MLP 후보 모델에 동일한 교차검증 과정을 적용한다.
- 검증 데이터에서 이벤트 리뷰일 확률이 0.5 이상이면 이벤트성 리뷰로 판단한다.
- 이후 각 모델과 fold별 성능 평가 지표를 저장하고, 모델별 평균 성능을 계산한다.
- 기존 `proposed_mlp_*` 산출물이 모두 있고 schema가 맞으면 이 단계는 CSV를 로드해 재사용한다.

저장하는 지표는 다음과 같다.

1. f1: precision과 recall의 균형을 나타내는 지표
2. pr_auc: 이벤트 리뷰처럼 관심 클래스의 탐지 성능을 볼 때 유용한 지표
3. precision: 이벤트 리뷰라고 예측한 것 중 실제 이벤트 리뷰의 비율
4. recall: 실제 이벤트 리뷰 중 모델이 이벤트 리뷰로 잡아낸 비율
5. roc_auc: 전체적인 이진 분류 구분 능력을 나타내는 지표

- 이벤트라고 찍은 예측의 신뢰도를 위해 F1을 우선으로 보고 PR-AUC를 보조 지표로 확인한다.
- 한 번의 split 결과에 의존하지 않고 train 내부 여러 fold 평균으로 설정을 고르기 위해서다.
- 지금 이 코드는 SMOTE를 학습 fold 안에서만 적용하면서 5-fold CV로 여러 MLP 하이퍼파라미터를 비교하고, validation 평균 성능이 가장 좋은 MLP 설정을 찾는 코드

### `cv_summary` 컬럼 설명

- `model`: MLP 후보 설정 이름
- `hidden_layer_sizes`: MLP 은닉층 구조
- `solver`: 가중치 업데이트 방식
- `learning_rate_init`: 초기 learning rate
- `alpha`: L2 정규화 강도
- `activation`: 은닉층 활성화 함수
- `batch_size`: 한 번에 학습하는 데이터 묶음 크기
- `pca_n_components`: PCA 설명 분산 보존 비율
- `smote_k_neighbors`: SMOTE 이웃 수
- `mean_f1`: 5개 fold의 평균 F1-score
- `std_f1`: 5개 fold F1-score의 표준편차
- `mean_pr_auc`: 5개 fold의 평균 PR-AUC
- `std_pr_auc`: 5개 fold PR-AUC의 표준편차
- `mean_precision`: 5개 fold의 평균 precision
- `mean_recall`: 5개 fold의 평균 recall
- `mean_roc_auc`: 5개 fold의 평균 ROC-AUC

In [4]:
if can_reuse_mlp:
    print('기존 04번 CV 결과를 재사용합니다.')
else:
    skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
    cv_rows = []

    for config in model_configs:
        for fold, (train_idx, valid_idx) in enumerate(skf.split(X_train_all, y_train_all), start=1):
            X_fold_train = X_train_all.iloc[train_idx]
            y_fold_train = y_train_all.iloc[train_idx]
            X_fold_valid = X_train_all.iloc[valid_idx]
            y_fold_valid = y_train_all.iloc[valid_idx]

            model = make_model(config, random_state=SEED + fold)
            model.fit(X_fold_train, y_fold_train)

            valid_prob = model.predict_proba(X_fold_valid)[:, 1]
            valid_pred = (valid_prob >= 0.5).astype(int)

            majority_count = int(y_fold_train.value_counts().max())
            cv_rows.append({
                'model': config['name'],
                'fold': fold,
                'hidden_layer_sizes': str(config['hidden_layer_sizes']),
                'solver': config['solver'],
                'learning_rate_init': config['learning_rate_init'],
                'alpha': config['alpha'],
                'activation': config['activation'],
                'batch_size': config['batch_size'],
                'pca_n_components': config['pca_n_components'],
                'smote_k_neighbors': config['smote_k_neighbors'],
                'train_before_label_0': int((y_fold_train == 0).sum()),
                'train_before_label_1': int((y_fold_train == 1).sum()),
                'train_after_smote_label_0': majority_count,
                'train_after_smote_label_1': majority_count,
                'valid_label_0': int((y_fold_valid == 0).sum()),
                'valid_label_1': int((y_fold_valid == 1).sum()),
                'f1': float(f1_score(y_fold_valid, valid_pred)),
                'pr_auc': float(average_precision_score(y_fold_valid, valid_prob)),
                'precision': float(precision_score(y_fold_valid, valid_pred, zero_division=0)),
                'recall': float(recall_score(y_fold_valid, valid_pred, zero_division=0)),
                'roc_auc': float(roc_auc_score(y_fold_valid, valid_prob)),
            })

    cv_results = pd.DataFrame(cv_rows)
    cv_summary = (
        cv_results.groupby([
            'model',
            'hidden_layer_sizes',
            'solver',
            'learning_rate_init',
            'alpha',
            'activation',
            'batch_size',
            'pca_n_components',
            'smote_k_neighbors',
        ])
        .agg(
            mean_f1=('f1', 'mean'),
            std_f1=('f1', 'std'),
            mean_pr_auc=('pr_auc', 'mean'),
            std_pr_auc=('pr_auc', 'std'),
            mean_precision=('precision', 'mean'),
            mean_recall=('recall', 'mean'),
            mean_roc_auc=('roc_auc', 'mean'),
        )
        .reset_index()
        .sort_values(['mean_f1', 'mean_pr_auc'], ascending=False)
    )

    cv_results.to_csv(PROPOSED_MLP_OUTPUTS['cv_results'], index=False, encoding='utf-8-sig')
    cv_summary.to_csv(PROPOSED_MLP_OUTPUTS['cv_summary'], index=False, encoding='utf-8-sig')

cv_summary

기존 04번 CV 결과를 재사용합니다.


,model,hidden_layer_sizes,solver,learning_rate_init,alpha,activation,batch_size,pca_n_components,smote_k_neighbors,mean_f1,std_f1,mean_pr_auc,std_pr_auc,mean_precision,mean_recall,mean_roc_auc
0,mlp_h32_tanh_sgd_lr0.001_alpha0.01_bs64_pca0p9...,"(32,)",sgd,0.0010,0.0100,tanh,64,0.9,5,0.463783,0.014785,0.416925,0.018587,0.408090,0.537415,0.568820
1,mlp_h32_tanh_sgd_lr0.001_alpha0.001_bs64_pca0p...,"(32,)",sgd,0.0010,0.0010,tanh,64,0.9,5,0.463777,0.014760,0.416841,0.018575,0.408084,0.537415,0.568714
2,mlp_h32_tanh_sgd_lr0.001_alpha0.0001_bs64_pca0...,"(32,)",sgd,0.0010,0.0001,tanh,64,0.9,5,0.462543,0.016065,0.414616,0.018924,0.406176,0.537415,0.566742
3,mlp_h64_tanh_sgd_lr0.0003_alpha0.01_bs64_pca0p...,"(64,)",sgd,0.0003,0.0100,tanh,64,0.9,5,0.461668,0.015128,0.414020,0.014890,0.399526,0.546939,0.565544
4,mlp_h64_tanh_sgd_lr0.0003_alpha0.0001_bs64_pca...,"(64,)",sgd,0.0003,0.0001,tanh,64,0.9,5,0.460501,0.015192,0.415096,0.015015,0.399997,0.542857,0.566447
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
187,mlp_h128_64_tanh_adam_lr0.001_alpha0.01_bs32_p...,"(128, 64)",adam,0.0010,0.0100,tanh,32,0.9,5,0.398498,0.014105,0.393886,0.017896,0.395577,0.401814,0.546404
188,mlp_h128_64_relu_sgd_lr0.001_alpha0.0001_bs64_...,"(128, 64)",sgd,0.0010,0.0001,relu,64,0.9,5,0.398370,0.024222,0.406463,0.008257,0.396484,0.400454,0.555997
189,mlp_h128_64_relu_adam_lr0.0003_alpha0.01_bs32_...,"(128, 64)",adam,0.0003,0.0100,relu,32,0.9,5,0.397299,0.042944,0.411105,0.014959,0.405289,0.390930,0.563128
190,mlp_h128_64_relu_adam_lr0.0003_alpha0.001_bs32...,"(128, 64)",adam,0.0003,0.0010,relu,32,0.9,5,0.395248,0.031337,0.408506,0.015041,0.403471,0.388209,0.560904


## 5. 최종 모델 학습 및 Validation/Test 평가

- 5-Fold 교차검증 결과에서 평균 F1-score와 PR-AUC 기준으로 가장 성능이 좋은 MLP 후보 모델을 선택한다.
    - 기존 찾은 최적의 모델과 모델 설정값을 후보 모델로 선택
- 선택된 모델 구조를 사용해 전체 train 데이터로 최종 모델을 학습한다.
- 또한 임계값은 validation 데이터에서만 조정하며, test 데이터는 모델 선택이 아닌 최종 일반화 성능 확인용으로만 사용한다.
- 최종 학습 단계에서도 클래스 불균형 보정을 위해 train 데이터에만 SMOTE를 적용한다.
- validation/test 데이터에는 SMOTE를 적용하지 않고, 원본 분포 그대로 평가한다.
- threshold는 일반 리뷰를 과도하게 이벤트 리뷰로 오탐하지 않도록 0.5 이상 후보에서만 탐색한다.
- 최종 모델과 평가 결과를 저장한다.


저장되는 결과는 다음과 같다.

1. `proposed_mlp_metrics.csv`: validation/test 성능 평가 결과
2. `proposed_mlp_metrics.json`: 선택된 모델 설정과 성능 평가 결과
3. `proposed_mlp_selected_config.json`: 최종 선택된 MLP 모델 설정
4. `proposed_mlp_final_model.joblib`: 최종 학습된 모델과 재사용에 필요한 설정

이유: CV에서 고른 설정을 전체 train에 다시 학습하고 validation/test는 최종 확인용으로 남기기 위해서다.


### 5-1. 최종 모델 선택 및 학습

- 교차검증 요약표에서 가장 위에 있는 후보 모델을 선택하고, 전체 train 데이터에 SMOTE를 적용한 뒤 최종 모델을 학습한다.

이유: 교차검증에서 선택한 설정만 최종 모델로 재학습해 별점 정제 단계에서 재사용할 bundle을 만들기 위해서다.


In [5]:
if can_reuse_mlp:
    selected = cached_selected_config
    best_model_name = selected['selected_model']
    best_config = {
        'name': selected['selected_model'],
        'hidden_layer_sizes': tuple(selected['hidden_layer_sizes']),
        'solver': selected['solver'],
        'learning_rate_init': selected['learning_rate_init'],
        'alpha': selected['alpha'],
        'activation': selected['activation'],
        'batch_size': selected['batch_size'],
        'pca_n_components': selected['pca_n_components'],
        'smote_k_neighbors': selected['smote_k_neighbors'],
    }
    final_model = cached_model_bundle['model']
    print(f'기존 최종 MLP 모델을 재사용합니다: {best_model_name}')
else:
    best_model_name = cv_summary.iloc[0]['model']
    best_config = next(config for config in model_configs if config['name'] == best_model_name)

    final_model = make_model(best_config, random_state=SEED)
    final_model.fit(X_train_all, y_train_all)

val_prob = final_model.predict_proba(X_val)[:, 1]
test_prob = final_model.predict_proba(X_test)[:, 1]

기존 최종 MLP 모델을 재사용합니다: mlp_h32_tanh_sgd_lr0.001_alpha0.01_bs64_pca0p9_smote5


### 5-2. Threshold 탐색

- 기본 threshold는 0.5이다. 
- 추가로 validation 데이터에서 F1-score가 가장 높은 threshold를 찾되, 너무 낮은 기준이 선택되지 않도록 후보를 0.5 이상으로 제한한다.

이유: threshold는 운영상 이벤트 판정 민감도를 바꾸므로 test가 아니라 validation에서만 조정해야 한다.


In [6]:
MIN_TUNED_THRESHOLD = 0.5

precisions, recalls, thresholds = precision_recall_curve(y_val, val_prob)

if len(thresholds) == 0:
    best_threshold = MIN_TUNED_THRESHOLD
    threshold_candidates = pd.DataFrame()
else:
    f1s = 2 * precisions[:-1] * recalls[:-1] / (precisions[:-1] + recalls[:-1] + 1e-8)
    threshold_candidates = pd.DataFrame({
        'threshold': thresholds,
        'precision': precisions[:-1],
        'recall': recalls[:-1],
        'f1': f1s,
    })
    threshold_candidates = threshold_candidates[threshold_candidates['threshold'] >= MIN_TUNED_THRESHOLD]

    if threshold_candidates.empty:
        best_threshold = MIN_TUNED_THRESHOLD
    else:
        best_threshold = float(
            threshold_candidates.sort_values('f1', ascending=False).iloc[0]['threshold']
        )

if can_reuse_mlp:
    best_threshold = float(selected.get('best_threshold_from_validation', best_threshold))

threshold_rank = threshold_candidates.sort_values('f1', ascending=False).head(10).reset_index(drop=True)
threshold_rank.insert(0, 'rank', threshold_rank.index + 1)
threshold_rank

,rank,threshold,precision,recall,f1
0,1,0.500871,0.386646,0.527542,0.446237
1,2,0.500368,0.386047,0.527542,0.445837
2,3,0.500878,0.385692,0.525424,0.444843
3,4,0.501094,0.384735,0.523305,0.443447
4,5,0.502488,0.385580,0.521186,0.443243
5,6,0.502949,0.386435,0.519068,0.443038
6,7,0.502354,0.384977,0.521186,0.442844
7,8,0.502829,0.385827,0.519068,0.442638
8,9,0.501857,0.384375,0.521186,0.442446
9,10,0.502783,0.385220,0.519068,0.442238


### 5-3. 성능 계산 및 결과 저장

- 기본 threshold 0.5와 validation에서 조정한 threshold를 비교한 후 모델 선택은 CV 결과와 validation threshold 기준으로 수행한다.
- 이후 성능표, 선택된 설정, 최종 학습 모델을 저장한다.
- 여기서 test 성능은 최종 보고용 지표로만 저장한다.

이유: 모델 설정과 성능을 파일로 남겨 05번 이후 비교와 재현에 같은 기준을 사용하기 위해서다.


In [7]:
def metric_row(split: str, y_true, prob, threshold: float) -> dict:
    pred = (prob >= threshold).astype(int)
    cm = confusion_matrix(y_true, pred, labels=[0, 1])
    return {
        'split': split,
        'threshold': float(threshold),
        'f1': float(f1_score(y_true, pred)),
        'pr_auc': float(average_precision_score(y_true, prob)),
        'precision': float(precision_score(y_true, pred, zero_division=0)),
        'recall': float(recall_score(y_true, pred, zero_division=0)),
        'roc_auc': float(roc_auc_score(y_true, prob)),
        'tn': int(cm[0, 0]),
        'fp': int(cm[0, 1]),
        'fn': int(cm[1, 0]),
        'tp': int(cm[1, 1]),
    }


if can_reuse_mlp:
    metrics = cached_metrics.copy()
    selected = cached_selected_config
    best_threshold = float(selected.get('best_threshold_from_validation', best_threshold))
    model_bundle = cached_model_bundle
    print('기존 04번 metrics/config/model bundle을 재사용합니다.')
else:
    metrics = pd.DataFrame([
        metric_row('validation_default_0_5', y_val, val_prob, 0.5),
        metric_row('validation_tuned_min_0_5', y_val, val_prob, best_threshold),
        metric_row('test_default_0_5', y_test, test_prob, 0.5),
        metric_row('test_tuned_min_0_5', y_test, test_prob, best_threshold),
    ])

    selected = {
        'selected_model': best_config['name'],
        'hidden_layer_sizes': list(best_config['hidden_layer_sizes']),
        'solver': best_config['solver'],
        'learning_rate_init': best_config['learning_rate_init'],
        'alpha': best_config['alpha'],
        'activation': best_config['activation'],
        'batch_size': best_config['batch_size'],
        'pca_n_components': best_config['pca_n_components'],
        'smote_k_neighbors': best_config['smote_k_neighbors'],
        'n_splits': N_SPLITS,
        'cv_grid_size': CV_GRID_SIZE,
        'selection_rule': 'highest mean_f1, then mean_pr_auc on leakage-safe 5-fold CV',
        'min_threshold_for_tuning': MIN_TUNED_THRESHOLD,
        'best_threshold_from_validation': best_threshold,
        'excluded_features': ['rating'],
    }

    metrics.to_csv(PROPOSED_MLP_OUTPUTS['metrics_csv'], index=False, encoding='utf-8-sig')
    with open(PROPOSED_MLP_OUTPUTS['metrics_json'], 'w', encoding='utf-8') as f:
        json.dump({'selected_config': selected, 'metrics': metrics.to_dict(orient='records')}, f, ensure_ascii=False, indent=2)
    with open(PROPOSED_MLP_OUTPUTS['selected_config'], 'w', encoding='utf-8') as f:
        json.dump(selected, f, ensure_ascii=False, indent=2)

    model_bundle = {
        'model': final_model,
        'feature_cols': feature_cols,
        'emb_cols': emb_cols,
        'meta_cols': meta_cols,
        'target_col': TARGET_COL,
        'selected_config': selected,
        'input_type': 'tabular_raw',
        'default_threshold': 0.5,
        'best_threshold': best_threshold,
    }
    joblib.dump(model_bundle, PROPOSED_MLP_OUTPUTS['model'])

기존 04번 metrics/config/model bundle을 재사용합니다.


### 5-4. 결과 표 출력

- 저장된 성능 결과를 노트북에서 바로 읽을 수 있도록 표 형태로 출력한다.

이유: 노트북 실행자가 저장 파일을 열지 않아도 선택 설정과 성능을 바로 확인할 수 있게 하기 위해서다.


In [8]:
selected_display = pd.DataFrame([{
    '선택 모델': selected['selected_model'],
    '은닉층 구조': str(tuple(selected['hidden_layer_sizes'])),
    'optimizer': selected['solver'],
    'learning_rate_init': selected['learning_rate_init'],
    'alpha': selected['alpha'],
    'activation': selected['activation'],
    'batch_size': selected['batch_size'],
    'PCA 보존율': selected['pca_n_components'],
    'SMOTE k_neighbors': selected['smote_k_neighbors'],
    'CV 후보 수': selected['cv_grid_size'],
    'fold 수': selected['n_splits'],
    '선택 기준': selected['selection_rule'],
    'threshold 최소 기준': selected['min_threshold_for_tuning'],
    'validation 최적 threshold': selected['best_threshold_from_validation'],
}])

threshold_display = pd.DataFrame([
    {'기준': 'default', 'threshold': 0.5, '설명': '확률이 0.5 이상이면 이벤트 리뷰로 예측'},
    {'기준': 'tuned_min_0_5', 'threshold': best_threshold, '설명': '0.5 이상 후보 중 validation F1-score가 가장 높았던 기준'},
])

metrics_display = metrics.rename(columns={
    'split': '평가 구분',
    'threshold': 'threshold',
    'f1': 'F1',
    'pr_auc': 'PR-AUC',
    'precision': 'Precision',
    'recall': 'Recall',
    'roc_auc': 'ROC-AUC',
    'tn': 'TN(일반→일반)',
    'fp': 'FP(일반→이벤트)',
    'fn': 'FN(이벤트→일반)',
    'tp': 'TP(이벤트→이벤트)',
})
metrics_display = metrics_display.round({
    'threshold': 4,
    'F1': 4,
    'PR-AUC': 4,
    'Precision': 4,
    'Recall': 4,
    'ROC-AUC': 4,
})

test_pred_tuned = (test_prob >= best_threshold).astype(int)
report_display = (
    pd.DataFrame(classification_report(
        y_test,
        test_pred_tuned,
        digits=4,
        zero_division=0,
        output_dict=True,
    ))
    .T
    .reset_index()
    .rename(columns={'index': '구분', 'precision': 'Precision', 'recall': 'Recall', 'f1-score': 'F1', 'support': '개수'})
)
report_display['구분'] = report_display['구분'].replace({
    '0': '일반 리뷰(0)',
    '1': '이벤트 리뷰(1)',
    'accuracy': '정확도',
    'macro avg': '단순 평균',
    'weighted avg': '가중 평균',
})
report_display = report_display.round({'Precision': 4, 'Recall': 4, 'F1': 4, '개수': 0})

print('최종 선택 모델')
display(selected_display)
print('threshold 기준')
display(threshold_display)
print('Validation/Test 성능 비교')
display(metrics_display)
print('Test 데이터 상세 리포트(임계값이 0.5이상 후보 중 F1이 가장 높은 threshold 기준)')
display(report_display)


최종 선택 모델


,선택 모델,은닉층 구조,optimizer,learning_rate_init,alpha,activation,batch_size,PCA 보존율,SMOTE k_neighbors,CV 후보 수,fold 수,선택 기준,threshold 최소 기준,validation 최적 threshold
0,mlp_h32_tanh_sgd_lr0.001_alpha0.01_bs64_pca0p9...,"(32,)",sgd,0.001,0.01,tanh,64,0.9,5,192,5,"highest mean_f1, then mean_pr_auc on leakage-s...",0.5,0.500871


threshold 기준


,기준,threshold,설명
0,default,0.500000,확률이 0.5 이상이면 이벤트 리뷰로 예측
1,tuned_min_0_5,0.500871,0.5 이상 후보 중 validation F1-score가 가장 높았던 기준


Validation/Test 성능 비교


,평가 구분,threshold,F1,PR-AUC,Precision,Recall,ROC-AUC,TN(일반→일반),FP(일반→이벤트),FN(이벤트→일반),TP(이벤트→이벤트)
0,validation_default_0_5,0.5000,0.4458,0.3850,0.3860,0.5275,0.5475,458,396,223,249
1,validation_tuned_min_0_5,0.5009,0.4462,0.3850,0.3866,0.5275,0.5475,459,395,223,249
2,test_default_0_5,0.5000,0.4407,0.4029,0.3973,0.4947,0.5547,499,355,239,234
3,test_tuned_min_0_5,0.5009,0.4405,0.4029,0.3983,0.4926,0.5547,502,352,240,233


Test 데이터 상세 리포트(임계값이 0.5이상 후보 중 F1이 가장 높은 threshold 기준)


,구분,Precision,Recall,F1,개수
0,일반 리뷰(0),0.6765,0.5878,0.6291,854.0
1,이벤트 리뷰(1),0.3983,0.4926,0.4405,473.0
2,정확도,0.5539,0.5539,0.5539,1.0
3,단순 평균,0.5374,0.5402,0.5348,1327.0
4,가중 평균,0.5774,0.5539,0.5618,1327.0


## 6. 결과 해석 및 다음 단계 연결

04번의 목적은 최종 프로젝트 모델을 확정하는 것이 아니라, 현재 제안 방식인 `KcBERT PCA feature + metadata + MLP` 구조를 학습하고 후보 모델 중 가장 나은 모델을 저장하는 것이다.

### 6-1. 현재 결과 해석

- 5-Fold 교차검증 기준으로 가장 높은 평균 F1을 보인 설정을 선택한다.
- 선택 기준은 `mean_f1`을 우선으로 보고, 동률 또는 유사한 경우 `mean_pr_auc`를 보조 기준으로 보는 방식이다.
- validation 데이터에서 threshold 후보를 0.5 이상으로 제한한 뒤 F1이 가장 높은 값을 찾는다.
- test 데이터 기준 성능은 모델 선택 이후 최종 확인용으로만 해석한다.
- 즉, 현재 Proposed MLP는 이벤트 리뷰 탐지를 위한 후보 모델이며, 실행 결과 수치는 노트북을 재실행한 뒤 출력 표를 기준으로 해석한다.

### 6-2. 현재 모델의 의미

- `outputs/proposed_mlp_final_model.joblib`에는 현재 학습된 Proposed MLP 모델과 입력 feature 컬럼, target 컬럼, 선택된 설정, threshold 정보가 함께 저장된다.
- 이 모델은 별점 정제 단계에서 바로 불러와 사용할 수 있는 후보 모델이다.
- 다만 아직 베이스라인과 비교하지 않았으므로, 프로젝트의 최종 모델로 확정된 것은 아니다.
- 따라서 04번의 결과는 `최종 모델 확정`이 아니라 `Proposed MLP 후보 모델 학습 완료`로 해석해야 한다.

### 6-3. 05번과의 연결

05번에서는 현재 Proposed MLP가 실제로 좋은 모델인지 확인하기 위해 베이스라인 모델과 비교해야 한다. 이 단계의 핵심은 단순히 점수만 비교하는 것이 아니라, `KcBERT 임베딩 결과에 메타데이터를 결합한 전략이 탐지율을 얼마나 향상시켰는지`를 검증하는 것이다.
비교해야 할 모델은 다음과 같다.

1. `TF-IDF + Random Forest`
2. `KcBERT PCA only + MLP`
3. `Metadata only + MLP`
4. `KcBERT PCA + Metadata Hybrid + MLP`

비교 지표는 04번과 동일하게 F1, PR-AUC, Precision, Recall, ROC-AUC를 사용한다. 이렇게 해야 04번의 Proposed MLP 결과와 05번의 baseline 결과를 같은 기준으로 비교할 수 있기 때문이다.

05번에서 확인할 내용은 다음과 같다.

- Hybrid MLP가 TF-IDF + Random Forest보다 좋은지 확인한다.
- Hybrid MLP가 KcBERT PCA only 모델보다 좋은지 확인한다.
- metadata only 모델이 어느 정도 성능을 내는지 확인한다.
- 메타데이터를 결합했을 때 F1, PR-AUC, Recall이 실제로 개선되는지 확인한다.
- 모델 예측에 큰 영향을 준 메타데이터가 무엇인지 분석한다.

### 6-4. 06번과의 연결: 오답 분석 및 최종 모델 선택

- 06번에서는 04번의 Proposed MLP 결과와 05번의 baseline 결과를 비교하고, 오답 리뷰를 따로 추출해 원인을 분석한다.
- 일반 리뷰를 이벤트 리뷰로 잘못 예측한 FP 사례와 이벤트 리뷰를 일반 리뷰로 놓친 FN 사례를 나눠서 본다.
- KcBERT가 은어, 신조어, 우회 표현을 처리할 때 어떤 한계를 보이는지 확인한다.
- 메타데이터가 텍스트 기반 모호성을 줄이는 데 실제로 도움을 줬는지 확인한다.
- 최종 모델 선택 기준은 이벤트 리뷰(1)의 F1과 PR-AUC를 중심으로 두고, precision/recall 균형과 오답 분석 결과도 함께 본다.

### 6-5. 07번과의 연결: 최종 모델 기반 별점 정제

- 07번에서는 06번에서 선택된 최종 모델을 불러와 전체 리뷰 데이터에 이벤트 리뷰 확률을 예측한다.
- 그 예측 결과를 이용해 이벤트성 리뷰로 판단된 행을 찾고, 별점 정제 전/후를 비교한다.
- 따라서 04번에서는 전체 리뷰 예측을 미리 수행하지 않고, 모델과 평가 결과만 저장하는 것이 맞다.

정리하면, 04번은 `Proposed MLP 학습 및 저장`까지 완료된 상태이며, 다음 단계는 베이스라인 비교, 메타데이터 효과 검증, 오답 분석을 끝낸 뒤 `최종 모델을 선택`하는 것이다.

이유: 04번 결과가 최종 결론이 아니라 baseline 비교와 오답 분석으로 이어지는 중간 산출물임을 명확히 하기 위해서다.
